# Table Visualization

This section demonstrates visualization of tabular data using the [Styler][styler]
class. For information on visualization with charting please see [Chart Visualization][viz]. This document is written as a Jupyter Notebook, and can be viewed or downloaded [here][download].

## Styler Object and Customising the Display
Styling and output display customisation should be performed **after** the data in a DataFrame has been processed. The Styler is **not** dynamically updated if further changes to the DataFrame are made. The `DataFrame.style` attribute is a property that returns a [Styler][styler] object. It has a `_repr_html_` method defined on it so it is rendered automatically in Jupyter Notebook.

The Styler, which can be used for large data but is primarily designed for small data, currently has the ability to output to these formats:

  - HTML
  - LaTeX
  - String (and CSV by extension)
  - Excel
  - (JSON is not currently available)

The first three of these have display customisation methods designed to format and customise the output. These include:

  - Formatting values, the index and columns headers, using [.format()][formatfunc] and [.format_index()][formatfuncindex],
  - Renaming the index or column header labels, using [.relabel_index()][relabelfunc]
  - Hiding certain columns, the index and/or column headers, or index names, using [.hide()][hidefunc]
  - Concatenating similar DataFrames, using [.concat()][concatfunc]
  
[styler]: ../reference/api/pandas.io.formats.style.Styler.rst
[viz]: visualization.rst
[download]: https://nbviewer.org/github/pandas-dev/pandas/blob/main/doc/source/user_guide/style.ipynb
[format]: https://docs.python.org/3/library/string.html#format-specification-mini-language
[formatfunc]: ../reference/api/pandas.io.formats.style.Styler.format.rst
[formatfuncindex]: ../reference/api/pandas.io.formats.style.Styler.format_index.rst
[relabelfunc]: ../reference/api/pandas.io.formats.style.Styler.relabel_index.rst
[hidefunc]: ../reference/api/pandas.io.formats.style.Styler.hide.rst
[concatfunc]: ../reference/api/pandas.io.formats.style.Styler.concat.rst

## Formatting the Display

### Formatting Values

The [Styler][styler] distinguishes the *display* value from the *actual* value, in both data values and index or columns headers. To control the display value, the text is printed in each cell as a string, and we can use the [.format()][formatfunc] and [.format_index()][formatfuncindex] methods to manipulate this according to a [format spec string][format] or a callable that takes a single value and returns a string. It is possible to define this for the whole table, or index, or for individual columns, or MultiIndex levels. We can also overwrite index names.

Additionally, the format function has a **precision** argument to specifically help format floats, as well as **decimal** and **thousands** separators to support other locales, an **na_rep** argument to display missing data, and an **escape** and **hyperlinks** arguments to help displaying safe-HTML or safe-LaTeX. The default formatter is configured to adopt pandas' global options such as `styler.format.precision` option, controllable using `with pd.option_context('format.precision', 2):`

[styler]: ../reference/api/pandas.io.formats.style.Styler.rst
[format]: https://docs.python.org/3/library/string.html#format-specification-mini-language
[formatfunc]: ../reference/api/pandas.io.formats.style.Styler.format.rst
[formatfuncindex]: ../reference/api/pandas.io.formats.style.Styler.format_index.rst
[relabelfunc]: ../reference/api/pandas.io.formats.style.Styler.relabel_index.rst

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame(
    {"strings": ["Adam", "Mike"], "ints": [1, 3], "floats": [1.123, 1000.23]}
)
df.style.format(precision=3, thousands=".", decimal=",").format_index(
    str.upper, axis=1
).relabel_index(["row 1", "row 2"], axis=0)

,STRINGS,INTS,FLOATS
row 1,Adam,1,"1,123"
row 2,Mike,3,"1.000,230"


Using Styler to manipulate the display is a useful feature because maintaining the indexing and data values for other purposes gives greater control. You do not have to overwrite your DataFrame to display it how you like. Here is a more comprehensive example of using the formatting functions whilst still relying on the underlying data for indexing and calculations.

In [2]:
weather_df = pd.DataFrame(
    np.random.default_rng().random((10, 2)) * 5,
    index=pd.date_range(start="2021-01-01", periods=10),
    columns=["Tokyo", "Beijing"],
)


def rain_condition(v):
    if v < 1.75:
        return "Dry"
    elif v < 2.75:
        return "Rain"
    return "Heavy Rain"


def make_pretty(styler):
    styler.set_caption("Weather Conditions")
    styler.format(rain_condition)
    styler.format_index(lambda v: v.strftime("%A"))
    styler.background_gradient(axis=None, vmin=1, vmax=5, cmap="YlGnBu")
    return styler


weather_df

,Tokyo,Beijing
2021-01-01,4.968788,3.763329
2021-01-02,0.437684,1.082656
2021-01-03,3.696452,3.415853
2021-01-04,3.228174,2.972024
2021-01-05,4.625737,0.497813
2021-01-06,4.875890,1.371619
2021-01-07,3.863812,3.574412
2021-01-08,4.527549,4.645622
2021-01-09,0.045759,3.462929
2021-01-10,3.600368,2.167409


In [3]:
weather_df.loc["2021-01-04":"2021-01-08"].style.pipe(make_pretty)

,Tokyo,Beijing
Monday,Heavy Rain,Heavy Rain
Tuesday,Heavy Rain,Dry
Wednesday,Heavy Rain,Dry
Thursday,Heavy Rain,Heavy Rain
Friday,Heavy Rain,Heavy Rain


### Hiding Data

The index and column headers can be completely hidden, as well subselecting rows or columns that one wishes to exclude. Both these options are performed using the same methods.

The index can be hidden from rendering by calling [.hide()][hideidx] without any arguments, which might be useful if your index is integer based. Similarly column headers can be hidden by calling [.hide(axis="columns")][hideidx] without any further arguments.

Specific rows or columns can be hidden from rendering by calling the same [.hide()][hideidx] method and passing in a row/column label, a list-like or a slice of row/column labels to for the ``subset`` argument.

Hiding does not change the integer arrangement of CSS classes, e.g. hiding the first two columns of a DataFrame means the column class indexing will still start at `col2`, since `col0` and `col1` are simply ignored.

[hideidx]: ../reference/api/pandas.io.formats.style.Styler.hide.rst

In [4]:
df = pd.DataFrame(np.random.default_rng().standard_normal((5, 5)))
df.style.hide(subset=[0, 2, 4], axis=0).hide(subset=[0, 2, 4], axis=1)

,1,3
1,0.584614,0.040719
3,-1.311074,0.264546


To invert the function to a **show** functionality it is best practice to compose a list of hidden items.

In [5]:
show = [0, 2, 4]
df.style.hide([row for row in df.index if row not in show], axis=0).hide(
    [col for col in df.columns if col not in show], axis=1
)

,0,2,4
0,0.905416,0.688778,-1.619478
2,-1.131221,-1.783136,-1.406076
4,0.862632,0.895874,0.181558


### Concatenating DataFrame Outputs

Two or more Stylers can be concatenated together provided they share the same columns. This is very useful for showing summary statistics for a DataFrame, and is often used in combination with DataFrame.agg.

Since the objects concatenated are Stylers they can independently be styled as will be shown below and their concatenation preserves those styles.

In [6]:
summary_styler = (
    df.agg(["sum", "mean"]).style.format(precision=3).relabel_index(["Sum", "Average"])
)
df.style.format(precision=1).concat(summary_styler)

,0,1,2,3,4
0,0.9,-0.7,0.7,-0.4,-1.6
1,-0.0,0.6,1.1,0.0,-1.1
2,-1.1,-1.1,-1.8,0.6,-1.4
3,-1.7,-1.3,0.1,0.3,1.2
4,0.9,-0.5,0.9,-1.1,0.2
Sum,-1.094,-3.052,0.980,-0.541,-2.703
Average,-0.219,-0.610,0.196,-0.108,-0.541


## Styler Object and HTML 

The [Styler][styler] was originally constructed to support the wide array of HTML formatting options. Its HTML output creates an HTML `<table>` and leverages CSS styling language to manipulate many parameters including colors, fonts, borders, background, etc. See [here][w3schools] for more information on styling HTML tables. This allows a lot of flexibility out of the box, and even enables web developers to integrate DataFrames into their existing user interface designs.

Below we demonstrate the default output, which looks very similar to the standard DataFrame HTML representation. But the HTML here has already attached some CSS classes to each cell, even if we haven't yet created any styles. We can view these by calling the  [.to_html()][tohtml] method, which returns the raw HTML as string, which is useful for further processing or adding to a file - read on in [More about CSS and HTML](#More-About-CSS-and-HTML). This section will also provide a walkthrough for how to convert this default output to represent a DataFrame output that is more communicative. For example how we can build `s`:

[tohtml]: ../reference/api/pandas.io.formats.style.Styler.to_html.rst

[styler]: ../reference/api/pandas.io.formats.style.Styler.rst
[w3schools]: https://www.w3schools.com/html/html_tables.asp

In [7]:
idx = pd.Index(["Tumour (Positive)", "Non-Tumour (Negative)"], name="Actual Label:")
cols = pd.MultiIndex.from_product(
    [["Decision Tree", "Regression", "Random"], ["Tumour", "Non-Tumour"]],
    names=["Model:", "Predicted:"],
)
df = pd.DataFrame(
    [[38.0, 2.0, 18.0, 22.0, 21, np.nan], [19, 439, 6, 452, 226, 232]],
    index=idx,
    columns=cols,
)
df.style

In [8]:
# Hidden cell to just create the below example: code is covered throughout the guide.
s = (
    df.style.hide([("Random", "Tumour"), ("Random", "Non-Tumour")], axis="columns")
    .format("{:.0f}")
    .set_table_styles(
        [
            {"selector": "", "props": "border-collapse: separate;"},
            {"selector": "caption", "props": "caption-side: bottom; font-size:1.3em;"},
            {
                "selector": ".index_name",
                "props": "font-style: italic; color: darkgrey; font-weight:normal;",
            },
            {
                "selector": "th:not(.index_name)",
                "props": "background-color: #000066; color: white;",
            },
            {"selector": "th.col_heading", "props": "text-align: center;"},
            {"selector": "th.col_heading.level0", "props": "font-size: 1.5em;"},
            {"selector": "th.col2", "props": "border-left: 1px solid white;"},
            {"selector": ".col2", "props": "border-left: 1px solid #000066;"},
            {"selector": "td", "props": "text-align: center; font-weight:bold;"},
            {"selector": ".true", "props": "background-color: #e6ffe6;"},
            {"selector": ".false", "props": "background-color: #ffe6e6;"},
            {"selector": ".border-red", "props": "border: 2px dashed red;"},
            {"selector": ".border-green", "props": "border: 2px dashed green;"},
            {"selector": "td:hover", "props": "background-color: #ffffb3;"},
        ]
    )
    .set_td_classes(
        pd.DataFrame(
            [
                ["true border-green", "false", "true", "false border-red", "", ""],
                ["false", "true", "false", "true", "", ""],
            ],
            index=df.index,
            columns=df.columns,
        )
    )
    .set_caption("Confusion matrix for multiple cancer prediction models.")
    .set_tooltips(
        pd.DataFrame(
            [
                [
                    "This model has a very strong true positive rate",
                    "",
                    "",
                    "This model's total number of false negatives is too high",
                    "",
                    "",
                ],
                ["", "", "", "", "", ""],
            ],
            index=df.index,
            columns=df.columns,
        ),
        css_class="pd-tt",
        props="visibility: hidden; "
        "position: absolute; z-index: 1; "
        "border: 1px solid #000066;"
        "background-color: white; color: #000066; font-size: 0.8em;"
        "transform: translate(0px, -24px); padding: 0.6em; border-radius: 0.5em;",
    )
)

In [9]:
s

The first step we have taken is the create the Styler object from the DataFrame and then select the range of interest by hiding unwanted columns with [.hide()][hideidx].

[hideidx]: ../reference/api/pandas.io.formats.style.Styler.hide.rst

In [10]:
s = df.style.format("{:.0f}").hide(
    [("Random", "Tumour"), ("Random", "Non-Tumour")], axis="columns"
)
s

In [11]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s.set_uuid("after_hide")

## Methods to Add Styles

There are **3 primary methods of adding custom CSS styles** to [Styler][styler]:

- Using [.set_table_styles()][table] to control broader areas of the table with specified internal CSS. Although table styles allow the flexibility to add CSS selectors and properties controlling all individual parts of the table, they are unwieldy for individual cell specifications. Also, note that table styles cannot be exported to Excel. 
- Using [.set_td_classes()][td_class] to directly link either external CSS classes to your data cells or link the internal CSS classes created by [.set_table_styles()][table]. See [here](#Setting-Classes-and-Linking-to-External-CSS). These cannot be used on column header rows or indexes, and also won't export to Excel. 
- Using the [.apply()][apply] and [.map()][map] functions to add direct internal CSS to specific data cells. See [here](#Styler-Functions). As of v1.4.0 there are also methods that work directly on column header rows or indexes: [.apply_index()][applyindex] and [.map_index()][mapindex]. Note that only these methods add styles that will export to Excel. These methods work in a similar way to [DataFrame.apply()][dfapply] and [DataFrame.map()][dfmap].

[table]: ../reference/api/pandas.io.formats.style.Styler.set_table_styles.rst
[styler]: ../reference/api/pandas.io.formats.style.Styler.rst
[td_class]: ../reference/api/pandas.io.formats.style.Styler.set_td_classes.rst
[apply]: ../reference/api/pandas.io.formats.style.Styler.apply.rst
[map]: ../reference/api/pandas.io.formats.style.Styler.map.rst
[applyindex]: ../reference/api/pandas.io.formats.style.Styler.apply_index.rst
[mapindex]: ../reference/api/pandas.io.formats.style.Styler.map_index.rst
[dfapply]: ../reference/api/pandas.DataFrame.apply.rst
[dfmap]: ../reference/api/pandas.DataFrame.map.rst

## Table Styles

Table styles are flexible enough to control all individual parts of the table, including column headers and indexes. 
However, they can be unwieldy to type for individual data cells or for any kind of conditional formatting, so we recommend that table styles are used for broad styling, such as entire rows or columns at a time.

Table styles are also used to control features which can apply to the whole table at once such as creating a generic hover functionality. The `:hover` pseudo-selector, as well as other pseudo-selectors, can only be used this way.

To replicate the normal format of CSS selectors and properties (attribute value pairs), e.g. 

```
tr:hover {
  background-color: #ffff99;
}
```

the necessary format to pass styles to [.set_table_styles()][table] is as a list of dicts, each with a CSS-selector tag and CSS-properties. Properties can either be a list of 2-tuples, or a regular CSS-string, for example:

[table]: ../reference/api/pandas.io.formats.style.Styler.set_table_styles.rst

In [12]:
cell_hover = {  # for row hover use <tr> instead of <td>
    "selector": "td:hover",
    "props": [("background-color", "#ffffb3")],
}
index_names = {
    "selector": ".index_name",
    "props": "font-style: italic; color: darkgrey; font-weight:normal;",
}
headers = {
    "selector": "th:not(.index_name)",
    "props": "background-color: #000066; color: white;",
}
s.set_table_styles([cell_hover, index_names, headers])

In [13]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s.set_uuid("after_tab_styles1")

Next we just add a couple more styling artifacts targeting specific parts of the table. Be careful here, since we are *chaining methods* we need to explicitly instruct the method **not to** ``overwrite`` the existing styles.

In [14]:
s.set_table_styles(
    [
        {"selector": "th.col_heading", "props": "text-align: center;"},
        {"selector": "th.col_heading.level0", "props": "font-size: 1.5em;"},
        {"selector": "td", "props": "text-align: center; font-weight: bold;"},
    ],
    overwrite=False,
)

In [15]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s.set_uuid("after_tab_styles2")

As a convenience method (*since version 1.2.0*) we can also pass a **dict** to [.set_table_styles()][table] which contains row or column keys. Behind the scenes Styler just indexes the keys and adds relevant `.col<m>` or `.row<n>` classes as necessary to the given CSS selectors.

[table]: ../reference/api/pandas.io.formats.style.Styler.set_table_styles.rst

In [16]:
s.set_table_styles(
    {
        ("Regression", "Tumour"): [
            {"selector": "th", "props": "border-left: 1px solid white"},
            {"selector": "td", "props": "border-left: 1px solid #000066"},
        ]
    },
    overwrite=False,
    axis=0,
)

In [17]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s.set_uuid("xyz01")

## Setting Classes and Linking to External CSS

If you have designed a website then it is likely you will already have an external CSS file that controls the styling of table and cell objects within it. You may want to use these native files rather than duplicate all the CSS in python (and duplicate any maintenance work).

### Table Attributes

It is very easy to add a `class` to the main `<table>` using [.set_table_attributes()][tableatt]. This method can also attach inline styles - read more in [CSS Hierarchies](#CSS-Hierarchies).

[tableatt]: ../reference/api/pandas.io.formats.style.Styler.set_table_attributes.rst

In [18]:
out = s.set_table_attributes('class="my-table-cls"').to_html()
print(out[out.find("<table") :][:109])

<table id="T_xyz01" class="my-table-cls">
  <thead>
    <tr>
      <th class="index_name level0" >Model:</th>


### Data Cell CSS Classes

*New in version 1.2.0*

The [.set_td_classes()][tdclass] method accepts a DataFrame with matching indices and columns to the underlying [Styler][styler]'s DataFrame. That DataFrame will contain strings as css-classes to add to individual data cells: the `<td>` elements of the `<table>`. Rather than use external CSS we will create our classes internally and add them to table style. We will save adding the borders until the [section on tooltips](#Tooltips-and-Captions).

[tdclass]: ../reference/api/pandas.io.formats.style.Styler.set_td_classes.rst
[styler]: ../reference/api/pandas.io.formats.style.Styler.rst

In [19]:
s.set_table_styles(
    [  # create internal CSS classes
        {"selector": ".true", "props": "background-color: #e6ffe6;"},
        {"selector": ".false", "props": "background-color: #ffe6e6;"},
    ],
    overwrite=False,
)
cell_color = pd.DataFrame(
    [["true ", "false ", "true ", "false "], ["false ", "true ", "false ", "true "]],
    index=df.index,
    columns=df.columns[:4],
)
s.set_td_classes(cell_color)

In [20]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s.set_uuid("after_classes")

## Styler Functions

### Acting on Data

We use the following methods to pass your style functions. Both of those methods take a function (and some other keyword arguments) and apply it to the DataFrame in a certain way, rendering CSS styles.

- [.map()][map] (elementwise): accepts a function that takes a single value and returns a string with the CSS attribute-value pair.
- [.apply()][apply] (column-/row-/table-wise): accepts a function that takes a Series or DataFrame and returns a Series, DataFrame, or numpy array with an identical shape where each element is a string with a CSS attribute-value pair. This method passes each column or row of your DataFrame one-at-a-time or the entire table at once, depending on the `axis` keyword argument. For columnwise use `axis=0`, rowwise use `axis=1`, and for the entire table at once use `axis=None`.

This method is powerful for applying multiple, complex logic to data cells. We create a new DataFrame to demonstrate this.

[apply]: ../reference/api/pandas.io.formats.style.Styler.apply.rst
[map]: ../reference/api/pandas.io.formats.style.Styler.map.rst

In [21]:
df2 = pd.DataFrame(
    np.random.default_rng(0).standard_normal((10, 4)), columns=["A", "B", "C", "D"]
)
df2.style

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


For example we can build a function that colors text if it is negative, and chain this with a function that partially fades cells of negligible value. Since this looks at each element in turn we use ``map``.

In [22]:
def style_negative(v, props=""):
    return props if v < 0 else None


s2 = df2.style.map(style_negative, props="color:red;").map(
    lambda v: "opacity: 20%;" if (v < 0.3) and (v > -0.3) else None
)
s2

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


In [23]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s2.set_uuid("after_applymap")

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


We can also build a function that highlights the maximum value across rows, cols, and the DataFrame all at once. In this case we use ``apply``. Below we highlight the maximum in a column.

In [24]:
def highlight_max(s, props=""):
    return np.where(s == np.nanmax(s.values), props, "")


s2.apply(highlight_max, props="color:white;background-color:darkblue", axis=0)

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


In [25]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s2.set_uuid("after_apply")

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


We can use the same function across the different axes, highlighting here the DataFrame maximum in purple, and row maximums in pink.

In [26]:
s2.apply(highlight_max, props="color:white;background-color:pink;", axis=1).apply(
    highlight_max, props="color:white;background-color:purple", axis=None
)

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


In [27]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s2.set_uuid("after_apply_again")

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


This last example shows how some styles have been overwritten by others. In general the most recent style applied is active but you can read more in the [section on CSS hierarchies](#CSS-Hierarchies). You can also apply these styles to more granular parts of the DataFrame - read more in section on [subset slicing](#Finer-Control-with-Slicing).

It is possible to replicate some of this functionality using just classes but it can be more cumbersome. See [item 3)  of Optimization](#Optimization)

<div class="alert alert-info">

*Debugging Tip*: If you're having trouble writing your style function, try just passing it into ``DataFrame.apply``. Internally, ``Styler.apply`` uses ``DataFrame.apply`` so the result should be the same, and with ``DataFrame.apply`` you will be able to inspect the CSS string output of your intended function in each cell.

</div>

### Acting on the Index and Column Headers

Similar application is achieved for headers by using:
    
- [.map_index()][mapindex] (elementwise): accepts a function that takes a single value and returns a string with the CSS attribute-value pair.
- [.apply_index()][applyindex] (level-wise): accepts a function that takes a Series and returns a Series, or numpy array with an identical shape where each element is a string with a CSS attribute-value pair. This method passes each level of your Index one-at-a-time. To style the index use `axis=0` and to style the column headers use `axis=1`.

You can select a `level` of a `MultiIndex` but currently no similar `subset` application is available for these methods.

[applyindex]: ../reference/api/pandas.io.formats.style.Styler.apply_index.rst
[mapindex]: ../reference/api/pandas.io.formats.style.Styler.map_index.rst

In [28]:
s2.map_index(lambda v: "color:pink;" if v > 4 else "color:darkblue;", axis=0)
s2.apply_index(
    lambda s: np.where(s.isin(["A", "B"]), "color:pink;", "color:darkblue;"), axis=1
)

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


## Tooltips and Captions

Table captions can be added with the [.set_caption()][caption] method. You can use table styles to control the CSS relevant to the caption.

[caption]: ../reference/api/pandas.io.formats.style.Styler.set_caption.rst

In [29]:
s.set_caption(
    "Confusion matrix for multiple cancer prediction models."
).set_table_styles(
    [{"selector": "caption", "props": "caption-side: bottom; font-size:1.25em;"}],
    overwrite=False,
)

In [30]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s.set_uuid("after_caption")

Adding tooltips (*since version 1.3.0*) can be done using the [.set_tooltips()][tooltips] method in the same way you can add CSS classes to data cells by providing a string based DataFrame with intersecting indices and columns. You don't have to specify a `css_class` name or any css `props` for the tooltips, since there are standard defaults, but the option is there if you want more visual control. 

[tooltips]: ../reference/api/pandas.io.formats.style.Styler.set_tooltips.rst

In [31]:
tt = pd.DataFrame(
    [
        [
            "This model has a very strong true positive rate",
            "This model's total number of false negatives is too high",
        ]
    ],
    index=["Tumour (Positive)"],
    columns=df.columns[[0, 3]],
)
s.set_tooltips(
    tt,
    props="visibility: hidden; position: absolute; z-index: 1; "
    "border: 1px solid #000066;"
    "background-color: white; color: #000066; font-size: 0.8em;"
    "transform: translate(0px, -24px); padding: 0.6em; "
    "border-radius: 0.5em;",
)

In [32]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s.set_uuid("after_tooltips")

The only thing left to do for our table is to add the highlighting borders to draw the audience attention to the tooltips. We will create internal CSS classes as before using table styles. **Setting classes always overwrites** so we need to make sure we add the previous classes.

In [33]:
s.set_table_styles(
    [  # create internal CSS classes
        {"selector": ".border-red", "props": "border: 2px dashed red;"},
        {"selector": ".border-green", "props": "border: 2px dashed green;"},
    ],
    overwrite=False,
)
cell_border = pd.DataFrame(
    [["border-green ", " ", " ", "border-red "], [" ", " ", " ", " "]],
    index=df.index,
    columns=df.columns[:4],
)
s.set_td_classes(cell_color + cell_border)

In [34]:
# Hidden cell to avoid CSS clashes and latter code upcoding previous formatting
s.set_uuid("after_borders")

## Finer Control with Slicing

The examples we have shown so far for the `Styler.apply` and `Styler.map` functions have not demonstrated the use of the ``subset`` argument. This is a useful argument which permits a lot of flexibility: it allows you to apply styles to specific rows or columns, without having to code that logic into your `style` function.

The value passed to `subset` behaves similar to slicing a DataFrame;

- A scalar is treated as a column label
- A list (or Series or NumPy array) is treated as multiple column labels
- A tuple is treated as `(row_indexer, column_indexer)`

Consider using `pd.IndexSlice` to construct the tuple for the last one. We will create a MultiIndexed DataFrame to demonstrate the functionality.

In [35]:
df3 = pd.DataFrame(
    np.random.default_rng().standard_normal((4, 4)),
    pd.MultiIndex.from_product([["A", "B"], ["r1", "r2"]]),
    columns=["c1", "c2", "c3", "c4"],
)
df3

c1        c2        c3        c4
A r1  0.986006  0.385101  1.249342  0.509096
  r2  0.412069 -0.269821 -0.074191  0.063617
B r1  0.288273  1.647746  0.229242  0.295998
  r2 -0.132955 -1.134032  0.044834 -0.610927

We will use subset to highlight the maximum in the third and fourth columns with red text. We will highlight the subset sliced region in yellow.

In [36]:
slice_ = ["c3", "c4"]
df3.style.apply(
    highlight_max, props="color:red;", axis=0, subset=slice_
).set_properties(**{"background-color": "#ffffb3"}, subset=slice_)

If combined with the ``IndexSlice`` as suggested then it can index across both dimensions with greater flexibility.

In [37]:
idx = pd.IndexSlice
slice_ = idx[idx[:, "r1"], idx["c2":"c4"]]
df3.style.apply(
    highlight_max, props="color:red;", axis=0, subset=slice_
).set_properties(**{"background-color": "#ffffb3"}, subset=slice_)

This also provides the flexibility to sub select rows when used with the `axis=1`.

In [38]:
slice_ = idx[idx[:, "r2"], :]
df3.style.apply(
    highlight_max, props="color:red;", axis=1, subset=slice_
).set_properties(**{"background-color": "#ffffb3"}, subset=slice_)

There is also scope to provide **conditional filtering**. 

Suppose we want to highlight the maximum across columns 2 and 4 only in the case that the sum of columns 1 and 3 is less than -2.0 *(essentially excluding rows* `(:,'r2')`*)*.

In [39]:
slice_ = idx[idx[(df3["c1"] + df3["c3"]) < -2.0], ["c2", "c4"]]
df3.style.apply(
    highlight_max, props="color:red;", axis=1, subset=slice_
).set_properties(**{"background-color": "#ffffb3"}, subset=slice_)

Only label-based slicing is supported right now, not positional, and not callables.

If your style function uses a `subset` or `axis` keyword argument, consider wrapping your function in a `functools.partial`, partialing out that keyword.

```python
my_func2 = functools.partial(my_func, subset=42)
```

## Optimization

Generally, for smaller tables and most cases, the rendered HTML does not need to be optimized, and we don't really recommend it. There are two cases where it is worth considering:

 - If you are rendering and styling a very large HTML table, certain browsers have performance issues.
 - If you are using ``Styler`` to dynamically create part of online user interfaces and want to improve network performance.
 
Here we recommend the following steps to implement:

### 1. Remove UUID and cell_ids

Ignore the `uuid` and set `cell_ids` to `False`. This will prevent unnecessary HTML.

<div class="alert alert-warning">

<font color=red>This is sub-optimal:</font>

</div>

In [40]:
df4 = pd.DataFrame([[1, 2], [3, 4]])
s4 = df4.style

<div class="alert alert-info">

<font color=green>This is better:</font>

</div>

In [41]:
from pandas.io.formats.style import Styler

s4 = Styler(df4, uuid_len=0, cell_ids=False)

### 2. Use table styles

Use table styles where possible (e.g. for all cells or rows or columns at a time) since the CSS is nearly always more efficient than other formats.

<div class="alert alert-warning">

<font color=red>This is sub-optimal:</font>

</div>

In [42]:
props = 'font-family: "Times New Roman", Times, serif; color: #e83e8c; font-size:1.3em;'
df4.style.map(lambda x: props, subset=[1])

,0,1
0,1,2
1,3,4


<div class="alert alert-info">

<font color=green>This is better:</font>

</div>

In [43]:
df4.style.set_table_styles([{"selector": "td.col1", "props": props}])

,0,1
0,1,2
1,3,4


### 3. Set classes instead of using Styler functions

For large DataFrames where the same style is applied to many cells it can be more efficient to declare the styles as classes and then apply those classes to data cells, rather than directly applying styles to cells. It is, however, probably still easier to use the Styler function api when you are not concerned about optimization.

<div class="alert alert-warning">

<font color=red>This is sub-optimal:</font>

</div>

In [44]:
df2.style.apply(
    highlight_max, props="color:white;background-color:darkblue;", axis=0
).apply(highlight_max, props="color:white;background-color:pink;", axis=1).apply(
    highlight_max, props="color:white;background-color:purple", axis=None
)

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


<div class="alert alert-info">

<font color=green>This is better:</font>

</div>

In [45]:
build = lambda x: pd.DataFrame(x, index=df2.index, columns=df2.columns)
cls1 = build(df2.apply(highlight_max, props="cls-1 ", axis=0))
cls2 = build(
    df2.apply(highlight_max, props="cls-2 ", axis=1, result_type="expand").values
)
cls3 = build(highlight_max(df2, props="cls-3 "))
df2.style.set_table_styles(
    [
        {"selector": ".cls-1", "props": "color:white;background-color:darkblue;"},
        {"selector": ".cls-2", "props": "color:white;background-color:pink;"},
        {"selector": ".cls-3", "props": "color:white;background-color:purple;"},
    ]
).set_td_classes(cls1 + cls2 + cls3)

,A,B,C,D
0,0.125730,-0.132105,0.640423,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,1.042513
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


### 4. Don't use tooltips

Tooltips require `cell_ids` to work and they generate extra HTML elements for *every* data cell.

### 5. If every byte counts use string replacement

You can remove unnecessary HTML, or shorten the default class names by replacing the default css dict. You can read a little more about CSS [below](#More-About-CSS-and-HTML).

In [46]:
my_css = {
    "row_heading": "",
    "col_heading": "",
    "index_name": "",
    "col": "c",
    "row": "r",
    "col_trim": "",
    "row_trim": "",
    "level": "l",
    "data": "",
    "blank": "",
}
html = Styler(df4, uuid_len=0, cell_ids=False)
html.set_table_styles(
    [
        {"selector": "td", "props": props},
        {"selector": ".c1", "props": "color:green;"},
        {"selector": ".l0", "props": "color:blue;"},
    ],
    css_class_names=my_css,
)
print(html.to_html())

<style type="text/css">
#T_ td {
  font-family: "Times New Roman", Times, serif;
  color: #e83e8c;
  font-size: 1.3em;
}
#T_ .c1 {
  color: green;
}
#T_ .l0 {
  color: blue;
}
</style>
<table id="T_">
  <thead>
    <tr>
      <th class=" l0" >&nbsp;</th>
      <th class=" l0 c0" >0</th>
      <th class=" l0 c1" >1</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th class=" l0 r0" >0</th>
      <td class=" r0 c0" >1</td>
      <td class=" r0 c1" >2</td>
    </tr>
    <tr>
      <th class=" l0 r1" >1</th>
      <td class=" r1 c0" >3</td>
      <td class=" r1 c1" >4</td>
    </tr>
  </tbody>
</table>



In [47]:
html

,0,1
0,1,2
1,3,4


## Builtin Styles

Some styling functions are common enough that we've "built them in" to the `Styler`, so you don't have to write them and apply them yourself. The current list of such functions is:

 - [.highlight_null][nullfunc]: for use with identifying missing data. 
 - [.highlight_min][minfunc] and [.highlight_max][maxfunc]: for use with identifying extremities in data.
 - [.highlight_between][betweenfunc] and [.highlight_quantile][quantilefunc]: for use with identifying classes within data.
 - [.background_gradient][bgfunc]: a flexible method for highlighting cells based on their, or other, values on a numeric scale.
 - [.text_gradient][textfunc]: similar method for highlighting text based on their, or other, values on a numeric scale.
 - [.bar][barfunc]: to display mini-charts within cell backgrounds.
 
The individual documentation on each function often gives more examples of their arguments.

[nullfunc]: ../reference/api/pandas.io.formats.style.Styler.highlight_null.rst
[minfunc]: ../reference/api/pandas.io.formats.style.Styler.highlight_min.rst
[maxfunc]: ../reference/api/pandas.io.formats.style.Styler.highlight_max.rst
[betweenfunc]: ../reference/api/pandas.io.formats.style.Styler.highlight_between.rst
[quantilefunc]: ../reference/api/pandas.io.formats.style.Styler.highlight_quantile.rst
[bgfunc]: ../reference/api/pandas.io.formats.style.Styler.background_gradient.rst
[textfunc]: ../reference/api/pandas.io.formats.style.Styler.text_gradient.rst
[barfunc]: ../reference/api/pandas.io.formats.style.Styler.bar.rst

### Highlight Null

In [48]:
df2.iloc[0, 2] = np.nan
df2.iloc[4, 3] = np.nan
df2.loc[:4].style.highlight_null(color="yellow")

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


### Highlight Min or Max

In [49]:
df2.loc[:4].style.highlight_max(
    axis=1, props=("color:white; font-weight:bold; background-color:darkblue;")
)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


### Highlight Between
This method accepts ranges as float, or NumPy arrays or Series provided the indexes match.

In [50]:
left = pd.Series([1.0, 0.0, 1.0], index=["A", "B", "D"])
df2.loc[:4].style.highlight_between(
    left=left, right=1.5, axis=1, props="color:white; background-color:purple;"
)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


### Highlight Quantile
Useful for detecting the highest or lowest percentile values

In [51]:
df2.loc[:4].style.highlight_quantile(q_left=0.85, axis=None, color="yellow")

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


### Background Gradient and Text Gradient

You can create "heatmaps" with the `background_gradient` and `text_gradient` methods. These require matplotlib, and we'll use [Seaborn](https://seaborn.pydata.org/) to get a nice colormap.

In [52]:
import seaborn as sns

cm = sns.light_palette("green", as_cmap=True)

df2.style.background_gradient(cmap=cm)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


In [53]:
df2.style.text_gradient(cmap=cm)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


[.background_gradient][bgfunc] and [.text_gradient][textfunc] have a number of keyword arguments to customise the gradients and colors. See the documentation.

[bgfunc]: ../reference/api/pandas.io.formats.style.Styler.background_gradient.rst
[textfunc]: ../reference/api/pandas.io.formats.style.Styler.text_gradient.rst

### Set properties

Use `Styler.set_properties` when the style doesn't actually depend on the values. This is just a simple wrapper for `.map` where the function returns the same properties for all cells.

In [54]:
df2.loc[:4].style.set_properties(
    **{"background-color": "black", "color": "lawngreen", "border-color": "white"}
)

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan


### Bar charts

You can include "bar charts" in your DataFrame.

In [55]:
df2.style.bar(subset=["A", "B"], color="#d65f5f")

,A,B,C,D
0,0.125730,-0.132105,nan,0.104900
1,-0.535669,0.361595,1.304000,0.947081
2,-0.703735,-1.265421,-0.623274,0.041326
3,-2.325031,-0.218792,-1.245911,-0.732267
4,-0.544259,-0.316300,0.411631,nan
5,-0.128535,1.366463,-0.665195,0.351510
6,0.903470,0.094012,-0.743499,-0.921725
7,-0.457726,0.220195,-1.009618,-0.209176
8,-0.159225,0.540846,0.214659,0.355373
9,-0.653829,-0.129614,0.783975,1.493431


Additional keyword arguments give more control on centering and positioning, and you can pass a list of `[color_negative, color_positive]` to highlight lower and higher values or a matplotlib colormap.

To showcase an example here's how you can change the above with the new `align` option, combined with setting `vmin` and `vmax` limits, the `width` of the figure, and underlying css `props` of cells, leaving space to display the text and the bars. We also use `text_gradient` to color the text the same as the bars using a matplotlib colormap (although in this case the visualization is probably better without this additional effect).

In [56]:
df2.style.format("{:.3f}", na_rep="").bar(
    align=0,
    vmin=-2.5,
    vmax=2.5,
    cmap="bwr",
    height=50,
    width=60,
    props="width: 120px; border-right: 1px solid black;",
).text_gradient(cmap="bwr", vmin=-2.5, vmax=2.5)

,A,B,C,D
0,0.126,-0.132,,0.105
1,-0.536,0.362,1.304,0.947
2,-0.704,-1.265,-0.623,0.041
3,-2.325,-0.219,-1.246,-0.732
4,-0.544,-0.316,0.412,
5,-0.129,1.366,-0.665,0.352
6,0.903,0.094,-0.743,-0.922
7,-0.458,0.220,-1.010,-0.209
8,-0.159,0.541,0.215,0.355
9,-0.654,-0.130,0.784,1.493


The following example aims to give a highlight of the behavior of the new align options:

In [57]:
# Hide the construction of the display chart from the user
import pandas as pd
from IPython.display import HTML

# Test series
test1 = pd.Series([-100, -60, -30, -20], name="All Negative")
test2 = pd.Series([-10, -5, 0, 90], name="Both Pos and Neg")
test3 = pd.Series([10, 20, 50, 100], name="All Positive")
test4 = pd.Series([100, 103, 101, 102], name="Large Positive")


head = """
<table>
    <thead>
        <th>Align</th>
        <th>All Negative</th>
        <th>Both Neg and Pos</th>
        <th>All Positive</th>
        <th>Large Positive</th>
    </thead>
    </tbody>

"""

aligns = ["left", "right", "zero", "mid", "mean", 99]
for align in aligns:
    row = "<tr><th>{}</th>".format(align)
    for series in [test1, test2, test3, test4]:
        s = series.copy()
        s.name = ""
        row += "<td>{}</td>".format(
            s.to_frame()
            .style.hide(axis="index")
            .bar(align=align, color=["#d65f5f", "#5fba7d"], width=100)
            .to_html()
        )  # testn['width']
    row += "</tr>"
    head += row

head += """
</tbody>
</table>"""

In [58]:
HTML(head)

left
""
-100
-60
-30
-20
""
-10
-5
0
90


## Sharing styles

Say you have a lovely style built up for a DataFrame, and now you want to apply the same style to a second DataFrame. Export the style with `df1.style.export`, and import it on the second DataFrame with `df1.style.set`

In [59]:
style1 = (
    df2.style.map(style_negative, props="color:red;")
    .map(lambda v: "opacity: 20%;" if (v < 0.3) and (v > -0.3) else None)
    .set_table_styles([{"selector": "th", "props": "color: blue;"}])
    .hide(axis="index")
)
style1

A,B,C,D
0.125730,-0.132105,nan,0.104900
-0.535669,0.361595,1.304000,0.947081
-0.703735,-1.265421,-0.623274,0.041326
-2.325031,-0.218792,-1.245911,-0.732267
-0.544259,-0.316300,0.411631,nan
-0.128535,1.366463,-0.665195,0.351510
0.903470,0.094012,-0.743499,-0.921725
-0.457726,0.220195,-1.009618,-0.209176
-0.159225,0.540846,0.214659,0.355373
-0.653829,-0.129614,0.783975,1.493431


In [60]:
style2 = df3.style
style2.use(style1.export())
style2

c1,c2,c3,c4
0.986006,0.385101,1.249342,0.509096
0.412069,-0.269821,-0.074191,0.063617
0.288273,1.647746,0.229242,0.295998
-0.132955,-1.134032,0.044834,-0.610927


Notice that you're able to share the styles even though they're data aware. The styles are re-evaluated on the new DataFrame they've been `use`d upon.

## Limitations

- DataFrame only (use `Series.to_frame().style`)
- The index and columns do not need to be unique, but certain styling functions can only work with unique indexes.
- No large repr, and construction performance isn't great; although we have some [HTML optimizations](#Optimization)
- You can only apply styles, you can't insert new HTML entities, except via subclassing.

## Other Fun and Useful Stuff

Here are a few interesting examples.

### Widgets

`Styler` interacts pretty well with widgets. If you're viewing this online instead of running the notebook yourself, you're missing out on interactively adjusting the color palette.

In [61]:
from ipywidgets import widgets


@widgets.interact
def f(h_neg=(0, 359, 1), h_pos=(0, 359), s=(0.0, 99.9), l_post=(0.0, 99.9)):
    return df2.style.background_gradient(
        cmap=sns.palettes.diverging_palette(
            h_neg=h_neg, h_pos=h_pos, s=s, l=l_post, as_cmap=True
        )
    )

interactive(children=(IntSlider(value=179, description='h_neg', max=359), IntSlider(value=179, description='h_…

### Magnify

In [62]:
def magnify():
    return [
        {"selector": "th", "props": [("font-size", "4pt")]},
        {"selector": "td", "props": [("padding", "0em 0em")]},
        {"selector": "th:hover", "props": [("font-size", "12pt")]},
        {
            "selector": "tr:hover td:hover",
            "props": [("max-width", "200px"), ("font-size", "12pt")],
        },
    ]

In [63]:
cmap = sns.diverging_palette(5, 250, as_cmap=True)
bigdf = pd.DataFrame(np.random.default_rng(25).standard_normal((20, 25))).cumsum()

bigdf.style.background_gradient(cmap, axis=1).set_properties(
    **{"max-width": "80px", "font-size": "1pt"}
).set_caption("Hover to magnify").format(precision=2).set_table_styles(magnify())

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
0,0.35,-0.00,-0.53,-2.28,0.02,0.93,1.04,-0.54,2.23,1.94,-0.23,-0.16,0.96,-0.25,0.22,1.22,1.19,-0.53,-0.46,-0.55,-0.01,0.07,-0.39,0.77,0.05
1,0.37,-1.97,-1.92,-2.97,0.38,0.06,2.60,-0.39,2.49,2.06,0.26,0.77,0.41,1.31,-0.55,1.74,0.98,-0.70,0.21,-0.35,1.59,-0.43,0.13,1.36,2.16
2,1.98,-1.27,-0.24,-2.12,-0.40,0.39,3.83,-0.91,4.01,2.42,-0.36,1.23,-1.16,0.62,-1.60,2.24,-0.04,-2.26,0.52,-0.83,0.87,-1.03,1.17,1.53,1.41
3,4.78,-0.78,1.67,-1.07,0.96,-0.48,3.26,-0.27,3.93,2.50,0.72,-1.07,-1.49,2.16,-2.07,2.28,0.34,-1.94,0.83,-1.76,1.20,-1.63,2.75,0.90,1.67
4,4.58,0.35,1.79,-0.36,0.40,-0.86,2.31,-0.44,3.65,1.36,-0.53,-0.91,-1.67,1.45,-2.99,1.17,0.44,-2.48,0.20,-2.32,0.73,-1.94,2.84,-0.86,1.41
5,3.38,-0.02,3.04,-2.28,-0.89,-1.38,0.01,1.95,3.01,2.98,-0.62,0.27,-0.86,1.15,-2.49,0.51,-0.04,-0.99,-0.67,-3.24,3.28,-2.33,3.14,0.78,0.92
6,2.81,1.80,3.02,-2.01,-1.23,-0.85,-1.57,2.12,4.01,3.94,-1.13,0.91,1.28,1.27,-1.81,0.15,-1.77,-0.25,-1.63,-3.94,3.34,-2.35,3.01,1.90,1.05
7,2.85,0.72,3.92,-3.29,-3.49,-1.67,-0.87,2.46,5.28,4.34,-0.76,1.24,0.64,0.81,-2.27,0.34,-2.71,-0.67,-1.35,-4.01,3.06,-2.55,1.78,0.54,0.25
8,3.80,1.51,4.19,-2.67,-2.71,-4.04,-1.42,3.30,7.04,4.52,0.96,2.40,0.78,-0.24,-3.28,0.28,-2.51,-0.74,-0.74,-3.44,3.11,-2.12,2.07,-1.35,-1.03
9,4.11,2.27,4.29,-2.97,-2.50,-3.58,-1.25,1.32,7.36,5.25,1.08,4.60,-0.64,-2.05,0.09,-0.72,-4.96,-1.31,0.09,-4.16,2.68,-1.37,4.02,-0.59,-0.72


### Sticky Headers

If you display a large matrix or DataFrame in a notebook, but you want to always see the column and row headers you can use the [.set_sticky][sticky] method which manipulates the table styles CSS.

[sticky]: ../reference/api/pandas.io.formats.style.Styler.set_sticky.rst

In [64]:
bigdf = pd.DataFrame(np.random.default_rng().standard_normal((16, 100)))
bigdf.style.set_sticky(axis="index")

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99
0,-1.017145,-0.833281,-0.013825,0.779651,-0.701345,-0.650684,-1.331855,0.812881,-0.220272,-2.009487,0.666090,0.658002,0.279832,-1.349077,-0.509725,0.361405,-1.140303,1.031344,-1.728391,-0.061667,-2.800072,0.354791,-0.749138,1.535707,0.207734,-1.724414,-0.598640,-0.789184,-0.535340,-0.820790,0.525406,1.110017,-0.597403,0.422047,-1.475016,-1.794687,0.276991,2.440729,0.449677,0.891157,-0.012202,0.194395,-0.303490,-1.211254,-0.394639,0.495442,-1.759557,-1.066839,0.555889,0.594681,0.673999,0.684114,2.527717,-0.952629,-3.361824,0.498717,2.013063,-0.765568,0.508900,0.147466,0.942602,0.568064,0.051625,-0.312928,0.433136,1.732381,1.550384,0.408094,-0.276721,0.331566,1.212183,-0.408982,-0.043256,-1.135747,-0.854204,1.268083,1.981530,1.329737,-0.654174,0.039469,-0.231947,0.436991,-0.428916,0.845390,0.731423,0.149585,0.886326,0.042174,0.387591,-1.218060,-0.026160,-0.722805,-0.196007,0.309206,-1.753097,1.002246,1.558684,0.269545,-0.230414,-0.200804
1,-2.107174,-2.075320,-1.317326,-1.135304,-0.766661,2.400456,0.798042,0.527101,1.051562,1.153289,-0.280140,0.875433,0.687952,-0.352768,-1.985305,0.177411,1.153402,-0.357220,-0.214672,1.225887,1.458867,1.678744,-0.548000,-0.117206,-0.188164,-0.071147,0.416495,0.549133,1.012973,-0.572648,-0.299205,-0.788906,-1.045898,1.051339,0.784297,-0.973342,-0.492208,1.306025,-0.590347,-0.271585,0.398106,-2.396230,-1.138649,0.445964,-0.541966,0.855540,0.407877,-1.383879,-0.344162,-0.109034,1.220540,-0.456040,0.039123,-0.652369,-0.846055,1.440243,-0.509938,-0.369841,-0.580711,0.261843,1.357114,0.634077,0.017308,-1.956417,-2.038248,-0.297216,0.812524,-0.457727,-0.935041,1.727045,0.294096,-1.340237,0.537948,-0.639695,0.574373,0.856673,0.165114,0.565881,0.357370,-1.692055,1.472855,-0.403703,0.518649,1.572314,-0.406881,0.522921,0.063984,0.138148,0.411502,0.435795,0.251960,-1.941827,1.468519,-0.578480,0.780210,0.317660,1.475215,-1.348276,0.755252,-0.020682
2,-0.883615,-2.098385,0.047109,0.475690,-1.653363,1.561936,1.618247,0.383749,1.692216,1.735671,-0.429876,-0.166500,-0.390887,-0.014698,-1.055817,-0.173456,1.571383,-0.475070,-0.477901,0.828787,-0.633198,0.523140,0.353202,-0.401292,-1.053575,0.592504,0.788733,0.946822,1.870423,0.541089,-1.563585,-1.554381,0.384425,-0.668080,0.429772,-0.190421,-0.044763,1.314644,-0.865929,1.632324,0.548694,-0.957215,0.393161,-2.556159,-0.255605,1.279768,-1.438752,-0.106286,-2.903263,0.424037,0.768202,-0.887271,-0.334972,0.734170,1.414619,0.593360,-0.121835,-0.459868,0.709889,0.219584,0.153443,0.601428,0.044660,-0.533042,0.304241,-0.368131,-1.255493,-0.064481,0.630633,-1.714956,0.384564,1.783164,0.204061,-1.121732,0.792715,-1.544428,-0.722990,0.417403,0.738327,1.592964,-0.121021,0.346192,1.013551,0.356596,0.419568,-1.139179,0.289882,-1.427222,-0.346271,-1.150926,-1.697315,0.332011,-1.131770,1.185347,0.423592,-0.022008,1.003747,-0.354893,0.159413,-0.924234
3,1.221128,-1.334635,1.557041,1.250923,0.264212,-0.321135,-0.116702,-2.391277,-0.760693,0.494716,-1.161080,-0.589768,0.080115,-1.084176,0.048871,-0.896858,-0.490953,-1.367848,0.761403,0.559173,0.377725,0.610099,0.284373,-0.788446,-0.018520,-1.210312,-0.028251,-0.246676,-1.038919,-1.597001,1.258430,0.242876,2.824370,1.455072,-0.108093,-0.300659,-0.959492,1.966608,0.327215,0.763412,0.512588,-0.882882,-0.169554,-0.430005,0.229784,0.172441,0.105890,0.137419,1.014598,-0.028859,-0.633808,-0.494146,-0.645265,0.409500,0.127770,0.857065,-0.979174,-1.222698,0.162981,-0.657222,0.157907,0.424012,0.458809,-0.225938,-0.479594,-0.330365,0.902438,-0.301010,0.361346,-0.475534,1.020556,0.673020,-0.403662,0.069773,-0.064314,0.515677,0.761071,-1.013794,-0.062293,-0.422310,0.295051,-0.542693,1.493579,0.602857,0.975719,0.172423,1.786213,0.269277,0.690247,1.149513,0.814

It is also possible to stick MultiIndexes and even only specific levels.

In [65]:
bigdf.index = pd.MultiIndex.from_product([["A", "B"], [0, 1], [0, 1, 2, 3]])
bigdf.style.set_sticky(axis="index", pixel_size=18, levels=[1, 2])

### HTML Escaping

Suppose you have to display HTML within HTML, that can be a bit of pain when the renderer can't distinguish. You can use the `escape` formatting option to handle this, and even use it within a formatter that contains HTML itself.

Note that if you're using `Styler` on untrusted, user-provided input to serve HTML then you should escape the input to prevent security vulnerabilities. See the Jinja2 documentation for more.

In [66]:
df4 = pd.DataFrame([["<div></div>", '"&other"', "<span></span>"]])
df4.style

,0,1,2
0,,"""&other""",


In [67]:
df4.style.format(escape="html")

,0,1,2
0,<div></div>,"""&other""",<span></span>


In [68]:
df4.style.format(
    '<a href="https://pandas.pydata.org" target="_blank">{}</a>', escape="html"
)

,0,1,2
0,<div></div>,"""&other""",<span></span>


## Export to Excel

Some support (*since version 0.20.0*) is available for exporting styled `DataFrames` to Excel worksheets using the `OpenPyXL` or `XlsxWriter` engines. CSS2.2 properties handled include:

- `background-color`
- `border-style` properties
- `border-width` properties
- `border-color` properties
- `color`
- `font-family`
- `font-style`
- `font-weight`
- `text-align`
- `text-decoration`
- `vertical-align`
- `white-space: nowrap`


- Shorthand and side-specific border properties are supported (e.g. `border-style` and `border-left-style`) as well as the `border` shorthands for all sides (`border: 1px solid green`) or specified sides (`border-left: 1px solid green`). Using a `border` shorthand will override any border properties set before it (See [CSS Working Group](https://drafts.csswg.org/css-backgrounds/#border-shorthands) for more details)


- Only CSS2 named colors and hex colors of the form `#rgb` or `#rrggbb` are currently supported.
- The following pseudo CSS properties are also available to set Excel specific style properties:
    - `number-format`
    - `border-style` (for Excel-specific styles: "hair", "mediumDashDot", "dashDotDot", "mediumDashDotDot", "dashDot", "slantDashDot", or "mediumDashed")

Table level styles, and data cell CSS-classes are not included in the export to Excel: individual cells must have their properties mapped by the `Styler.apply` and/or `Styler.map` methods.

In [69]:
df2.style.map(style_negative, props="color:red;").highlight_max(axis=0).to_excel(
    "styled.xlsx", engine="openpyxl"
)

A screenshot of the output:

![Excel spreadsheet with styled DataFrame](../_static/style-excel.png)


## Export to LaTeX

There is support (*since version 1.3.0*) to export `Styler` to LaTeX. The documentation for the [.to_latex][latex] method gives further detail and numerous examples.

[latex]: ../reference/api/pandas.io.formats.style.Styler.to_latex.rst

## More About CSS and HTML

Cascading Style Sheet (CSS) language, which is designed to influence how a browser renders HTML elements, has its own peculiarities. It never reports errors: it just silently ignores them and doesn't render your objects how you intend so can sometimes be frustrating. Here is a very brief primer on how ``Styler`` creates HTML and interacts with CSS, with advice on common pitfalls to avoid.

### CSS Classes and Ids

The precise structure of the CSS `class` attached to each cell is as follows.

- Cells with Index and Column names include `index_name` and `level<k>` where `k` is its level in a MultiIndex
- Index label cells include
  + `row_heading`
  + `level<k>` where `k` is the level in a MultiIndex
  + `row<m>` where `m` is the numeric position of the row
- Column label cells include
  + `col_heading`
  + `level<k>` where `k` is the level in a MultiIndex
  + `col<n>` where `n` is the numeric position of the column
- Data cells include
  + `data`
  + `row<m>`, where `m` is the numeric position of the cell.
  + `col<n>`, where `n` is the numeric position of the cell.
- Blank cells include `blank`
- Trimmed cells include `col_trim` or `row_trim`

The structure of the `id` is `T_uuid_level<k>_row<m>_col<n>` where `level<k>` is used only on headings, and headings will only have either `row<m>` or `col<n>` whichever is needed. By default we've also prepended each row/column identifier with a UUID unique to each DataFrame so that the style from one doesn't collide with the styling from another within the same notebook or page. You can read more about the use of UUIDs in [Optimization](#Optimization).

We can see example of the HTML by calling the [.to_html()][tohtml] method.

[tohtml]: ../reference/api/pandas.io.formats.style.Styler.to_html.rst

In [70]:
print(
    pd.DataFrame(
        [[1, 2], [3, 4]], index=["i1", "i2"], columns=["c1", "c2"]
    ).style.to_html()
)

<style type="text/css">
</style>
<table id="T_fc828">
  <thead>
    <tr>
      <th class="blank level0" >&nbsp;</th>
      <th id="T_fc828_level0_col0" class="col_heading level0 col0" >c1</th>
      <th id="T_fc828_level0_col1" class="col_heading level0 col1" >c2</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th id="T_fc828_level0_row0" class="row_heading level0 row0" >i1</th>
      <td id="T_fc828_row0_col0" class="data row0 col0" >1</td>
      <td id="T_fc828_row0_col1" class="data row0 col1" >2</td>
    </tr>
    <tr>
      <th id="T_fc828_level0_row1" class="row_heading level0 row1" >i2</th>
      <td id="T_fc828_row1_col0" class="data row1 col0" >3</td>
      <td id="T_fc828_row1_col1" class="data row1 col1" >4</td>
    </tr>
  </tbody>
</table>



### CSS Hierarchies

The examples have shown that when CSS styles overlap, the one that comes last in the HTML render, takes precedence. So the following yield different results:

In [71]:
df4 = pd.DataFrame([["text"]])
df4.style.map(lambda x: "color:green;").map(lambda x: "color:red;")

,0
0,text


In [72]:
df4.style.map(lambda x: "color:red;").map(lambda x: "color:green;")

,0
0,text


This is only true for CSS rules that are equivalent in hierarchy, or importance. You can read more about [CSS specificity here](https://www.w3schools.com/css/css_specificity.asp) but for our purposes it suffices to summarize the key points:

A CSS importance score for each HTML element is derived by starting at zero and adding:

 - 1000 for an inline style attribute
 - 100 for each ID
 - 10 for each attribute, class or pseudo-class
 - 1 for each element name or pseudo-element
 
Let's use this to describe the action of the following configurations

In [73]:
df4.style.set_uuid("a_").set_table_styles(
    [{"selector": "td", "props": "color:red;"}]
).map(lambda x: "color:green;")

,0
0,text


This text is red because the generated selector `#T_a_ td` is worth 101 (ID plus element), whereas `#T_a_row0_col0` is only worth 100 (ID), so is considered inferior even though in the HTML it comes after the previous.

In [74]:
df4.style.set_uuid("b_").set_table_styles(
    [
        {"selector": "td", "props": "color:red;"},
        {"selector": ".cls-1", "props": "color:blue;"},
    ]
).map(lambda x: "color:green;").set_td_classes(pd.DataFrame([["cls-1"]]))

,0
0,text


In the above case the text is blue because the selector `#T_b_ .cls-1` is worth 110 (ID plus class), which takes precedence.

In [75]:
df4.style.set_uuid("c_").set_table_styles(
    [
        {"selector": "td", "props": "color:red;"},
        {"selector": ".cls-1", "props": "color:blue;"},
        {"selector": "td.data", "props": "color:yellow;"},
    ]
).map(lambda x: "color:green;").set_td_classes(pd.DataFrame([["cls-1"]]))

,0
0,text


Now we have created another table style this time the selector `T_c_ td.data` (ID plus element plus class) gets bumped up to 111. 

If your style fails to be applied, and its really frustrating, try the `!important` trump card.

In [76]:
df4.style.set_uuid("d_").set_table_styles(
    [
        {"selector": "td", "props": "color:red;"},
        {"selector": ".cls-1", "props": "color:blue;"},
        {"selector": "td.data", "props": "color:yellow;"},
    ]
).map(lambda x: "color:green !important;").set_td_classes(pd.DataFrame([["cls-1"]]))

,0
0,text


Finally got that green text after all!

## Extensibility

The core of pandas is, and will remain, its "high-performance, easy-to-use data structures".
With that in mind, we hope that `DataFrame.style` accomplishes two goals

- Provide an API that is pleasing to use interactively and is "good enough" for many tasks
- Provide the foundations for dedicated libraries to build on

If you build a great library on top of this, let us know and we'll [link](https://pandas.pydata.org/community/ecosystem.html) to it.

### Subclassing

If the default template doesn't quite suit your needs, you can subclass Styler and extend or override the template.
We'll show an example of extending the default template to insert a custom header before each table.

In [77]:
from jinja2 import Environment, ChoiceLoader, FileSystemLoader
from IPython.display import HTML
from pandas.io.formats.style import Styler

We'll use the following template:

In [78]:
with open("templates/myhtml.tpl") as f_html:
    print(f_html.read())

{% extends "html_table.tpl" %}
{% block table %}
<h1>{{ table_title|default("My Table") }}</h1>
{{ super() }}
{% endblock table %}



Now that we've created a template, we need to set up a subclass of ``Styler`` that
knows about it.

In [79]:
class MyStyler(Styler):
    env = Environment(
        loader=ChoiceLoader(
            [
                FileSystemLoader("templates"),  # contains ours
                Styler.loader,  # the default
            ]
        )
    )
    template_html_table = env.get_template("myhtml.tpl")

Notice that we include the original loader in our environment's loader.
That's because we extend the original template, so the Jinja environment needs
to be able to find it.

Now we can use that custom styler. It's `__init__` takes a DataFrame.

In [80]:
MyStyler(df3)

Our custom template accepts a `table_title` keyword. We can provide the value in the `.to_html` method.

In [81]:
HTML(MyStyler(df3).to_html(table_title="Extending Example"))

For convenience, we provide the `Styler.from_custom_template` method that does the same as the custom subclass.

In [82]:
EasyStyler = Styler.from_custom_template("templates", "myhtml.tpl")
HTML(EasyStyler(df3).to_html(table_title="Another Title"))

#### Template Structure

Here's the template structure for the both the style generation template and the table generation template:

Style template:

In [83]:
with open("templates/html_style_structure.html") as f_sty:
    style_structure = f_sty.read()

In [84]:
HTML(style_structure)

Table template:

In [85]:
with open("templates/html_table_structure.html") as f_table_struct:
    table_structure = f_table_struct.read()

In [86]:
HTML(table_structure)

See the template in the [GitHub repo](https://github.com/pandas-dev/pandas) for more details.

In [87]:
# # Hack to get the same style in the notebook as the
# # main site. This is hidden in the docs.
# from IPython.display import HTML
# with open("themes/nature_with_gtoc/static/nature.css_t") as f:
#     css = f.read()

# HTML('<style>{}</style>'.format(css))